In [7]:
:set -XRankNTypes
:set -XScopedTypeVariables
putStrLn "Setup done."

Setup done.

# 🌐 Распределённые вычисления в Haskell

**Разделы:** Pi-исчисление → Cloud Haskell → CRDT → Eventual Consistency → MapReduce

**Уровень:** продвинутый | **Предварительные требования:** [Монады](Monads.ipynb), [Конкурентность](Concurrency.ipynb)

![Ландшафт распределённых вычислений](../diagrams/haskell/dist_landscape.svg)

---

> **📦 Зависимости**
> **Пакеты:** `containers`, `stm`
> **Расширения:** `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables))


❖Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | 1️⃣ Pi-исчисление: процессы как категория |  |
| 2 | 2️⃣ Cloud Haskell: Process монада |  |
| 3 | 3️⃣ CRDT: Eventually Consistent Data Structures |  |
| 4 | 4️⃣ MapReduce: функтор + монада |  |
| 5 | 5️⃣ Категорный взгняд: единая картина |  |

## 1️⃣ Pi-исчисление: процессы как категория

Pi-исчисление — математическая основа мобильных процессов. Категория **Proc**:

| Конструкция | Смысл | Haskell-аналог |
|-------------|-------|----------------|
| `P | Q` | Параллельная композиция | forkIO (концептуально) |
| `x(y).P` | Получить y по каналу x | receive :: Process a |
| `nu x.P` | Новый приватный канал x | newTChan |

**Категорный взгляд:** Тензорное произведение P x Q = P | Q (параллельность = моноидальная структура)

In [8]:
-- Pi-исчисление через STM-каналы
import Control.Concurrent.STM
import Control.Concurrent.STM.TQueue

-- Канал = TQueue (аналог nu x — создание нового канала)
type Channel a = TQueue a

newChannel :: STM (Channel a)
newChannel = newTQueue

-- Отправить по каналу (x-bar <y>)
sendCh :: Channel a -> a -> STM ()
sendCh = writeTQueue

-- Получить из канала (x(y))
recvCh :: Channel a -> STM a
recvCh = readTQueue

-- Демо: простой Pi-пайп (Producer -> Transformer -> Consumer)
demoPiPipeline :: IO ()
demoPiPipeline = do
  (ch1, ch2) <- atomically $ do
    c1 <- newChannel
    c2 <- newChannel
    return (c1, c2)
  
  -- Producer: записывает в ch1
  atomically $ mapM_ (sendCh ch1) [1..5 :: Int]
  
  -- Transformer: ch1 -> ch2, map (+10)
  atomically $ do
    vals <- sequence $ replicate 5 (recvCh ch1)
    mapM_ (sendCh ch2 . (+10)) vals
  
  -- Consumer: читает из ch2
  results <- atomically $ sequence $ replicate 5 (recvCh ch2)
  putStrLn $ "Pi-pipeline result: " ++ show results

demoPiPipeline

Line 32: Use replicateM
Found:
sequence $ replicate 5 (recvCh ch1)
Why not:
Control.Monad.replicateM 5 (recvCh ch1)Line 36: Use replicateM
Found:
sequence $ replicate 5 (recvCh ch2)
Why not:
Control.Monad.replicateM 5 (recvCh ch2)

Pi-pipeline result: [11,12,13,14,15]

## 2️⃣ Cloud Haskell: Process монада

**Cloud Haskell** реализует акторную модель. Концептуальный API:

```haskell
spawn  :: NodeId -> Closure (Process ()) -> Process ProcessId
send   :: Serializable a => ProcessId -> a -> Process ()
expect :: Serializable a => Process a
```

| Компонент | Категорный смысл |
|-----------|------------------|
| `Process a` | Монада эффектов с коммуникацией |
| `ProcessId` | Объект в категории акторов |
| `send` / `expect` | Морфизмы через каналы |
| `Closure a` | Сериализуемая функция |

Аналогия: **ActorCategory**, где объекты = акторы, морфизмы = сообщения.

In [9]:
-- Симуляция актор-паттерна через STM
import Control.Concurrent.STM
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map

-- Почтовый ящик актора
data ActorBox a = ActorBox
  { mailbox :: TQueue a
  , actorId :: Int
  }

newActor :: Int -> STM (ActorBox a)
newActor aid = do
  mb <- newTQueue
  return $ ActorBox mb aid

sendMsg :: ActorBox a -> a -> STM ()
sendMsg ab = writeTQueue (mailbox ab)

demoActorCounters :: IO ()
demoActorCounters = do
  (actor1, actor2) <- atomically $ do
    a1 <- newActor 1
    a2 <- newActor 2
    return (a1, a2)
  
  atomically $ do
    sendMsg actor1 (10 :: Int)
    sendMsg actor1 20
    sendMsg actor2 100
    sendMsg actor2 200
  
  sum1 <- atomically $ do
    v1 <- readTQueue (mailbox actor1)
    v2 <- readTQueue (mailbox actor1)
    return (v1 + v2 :: Int)
  
  sum2 <- atomically $ do
    v1 <- readTQueue (mailbox actor2)
    v2 <- readTQueue (mailbox actor2)
    return (v1 + v2 :: Int)
  
  putStrLn $ "Actor 1 sum: " ++ show sum1
  putStrLn $ "Actor 2 sum: " ++ show sum2
  putStrLn $ "Cluster total: " ++ show (sum1 + sum2)

demoActorCounters

Actor 1 sum: 30
Actor 2 sum: 300
Cluster total: 330

## 3️⃣ CRDT: Eventually Consistent Data Structures

**CRDT** — структуры данных, гарантирующие схождение без координации.

**Join-Semilattice** — категорная основа:
- `a join b = b join a` (коммутативность)
- `(a join b) join c = a join (b join c)` (ассоциативность)
- `a join a = a` (идемпотентность)

| Тип CRDT | Операция | Смысл |
|----------|---------|-------|
| **G-Counter** | max по векторам | Только инкремент |
| **PN-Counter** | G-Counter x2 | Инкремент + декремент |
| **G-Set** | объединение | Только добавление |

In [10]:
-- G-Counter (CRDT) в Haskell
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map
import qualified Data.Set as Set

newtype GCounter = GCounter { gcMap :: Map Int Int }
  deriving (Show, Eq)

emptyGCounter :: GCounter
emptyGCounter = GCounter Map.empty

increment :: Int -> Int -> GCounter -> GCounter
increment nodeId delta (GCounter m) =
  GCounter $ Map.insertWith (+) nodeId delta m

value :: GCounter -> Int
value (GCounter m) = Map.foldr (+) 0 m

merge :: GCounter -> GCounter -> GCounter
merge (GCounter m1) (GCounter m2) =
  GCounter $ Map.unionWith max m1 m2

demoGCounter :: IO ()
demoGCounter = do
  let node1 = increment 1 1 . increment 1 1 . increment 1 1 $ emptyGCounter
  let node2 = increment 2 1 . increment 2 1 $ emptyGCounter
  let node3 = increment 3 1 node1
  
  putStrLn $ "Node 1: " ++ show (value node1)
  putStrLn $ "Node 2: " ++ show (value node2)
  putStrLn $ "Node 3: " ++ show (value node3)
  
  let merged123 = merge node1 (merge node2 node3)
  let merged_rev = merge node3 (merge node2 node1)
  
  putStrLn $ "\nMerged (1+2+3): " ++ show (value merged123)
  putStrLn $ "Commutative: " ++ show (merged123 == merged_rev)
  putStrLn $ "Idempotent: " ++ show (merge merged123 merged123 == merged123)

demoGCounter

Node 1: 3
Node 2: 2
Node 3: 4

Merged (1+2+3): 6
Commutative: True
Idempotent: True

In [11]:
-- PN-Counter + G-Set
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map
import qualified Data.Set as Set

data PNCounter = PNCounter
  { incPart :: GCounter
  , decPart :: GCounter
  } deriving (Show, Eq)

emptyPN :: PNCounter
emptyPN = PNCounter emptyGCounter emptyGCounter

incPN :: Int -> PNCounter -> PNCounter
incPN nodeId (PNCounter i d) = PNCounter (increment nodeId 1 i) d

decPN :: Int -> PNCounter -> PNCounter
decPN nodeId (PNCounter i d) = PNCounter i (increment nodeId 1 d)

valuePN :: PNCounter -> Int
valuePN (PNCounter i d) = value i - value d

mergePN :: PNCounter -> PNCounter -> PNCounter
mergePN (PNCounter i1 d1) (PNCounter i2 d2) =
  PNCounter (merge i1 i2) (merge d1 d2)

newtype GSet a = GSet { gsSet :: Set.Set a } deriving (Show, Eq)

emptyGSet :: Ord a => GSet a
emptyGSet = GSet Set.empty

addGSet :: Ord a => a -> GSet a -> GSet a
addGSet x (GSet s) = GSet (Set.insert x s)

mergeGSet :: Ord a => GSet a -> GSet a -> GSet a
mergeGSet (GSet s1) (GSet s2) = GSet (Set.union s1 s2)

demoCRDTs :: IO ()
demoCRDTs = do
  let cart1 = addGSet "apple" . addGSet "bread" $ emptyGSet
  let cart2 = addGSet "milk"  . addGSet "bread" $ emptyGSet
  let cart3 = addGSet "eggs"  $ emptyGSet
  let finalCart = mergeGSet cart1 (mergeGSet cart2 cart3)
  putStrLn $ "Final cart: " ++ show (gsSet finalCart)
  
  let votes = incPN 1 . incPN 2 . incPN 3 . decPN 4 . decPN 5 $ emptyPN
  putStrLn $ "\nVotes: " ++ show (valuePN votes) ++ " (3 for, 2 against)"

demoCRDTs

Line 42: Redundant $
Found:
addGSet "eggs" $ emptyGSet
Why not:
addGSet "eggs" emptyGSet

Final cart: fromList ["apple","bread","eggs","milk"]

Votes: 1 (3 for, 2 against)

## 4️⃣ MapReduce: функтор + монада

**MapReduce** — парадигма распределённой обработки. Категорная структура:

```haskell
-- Map = функтор (fmap) на шардах
mapper :: a -> [(k, v)]

-- Shuffle = предел (colimit) по ключу
-- Reduce = монадическое связывание (>>=)
reducer :: k -> [v] -> r
```

| Шаг | Категорный объект | Смысл |
|------|------|------|
| Map | Функтор | Параллельная трансформация |
| Shuffle | Предел | Группировка по ключу |
| Reduce | Катаморфизм | Агрегация |

In [12]:
-- MapReduce в чистом Haskell
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map

mapStep :: (a -> [(k, v)]) -> [a] -> [(k, v)]
mapStep f = concatMap f

shuffleStep :: Ord k => [(k, v)] -> Map k [v]
shuffleStep = foldr (\(k, v) acc -> Map.insertWith (++) k [v] acc) Map.empty

reduceStep :: (k -> [v] -> r) -> Map k [v] -> Map k r
reduceStep f = Map.mapWithKey f

mapReduce :: Ord k => (a -> [(k, v)]) -> (k -> [v] -> r) -> [a] -> Map k r
mapReduce mapper reducer input =
  reduceStep reducer . shuffleStep . mapStep mapper $ input

-- Демо: подсчёт слов (Word Count)
wordCount :: [String] -> Map String Int
wordCount = mapReduce
  (\doc -> map (\w -> (w, 1)) (words doc))
  (\_ counts -> sum counts)

demoMapReduce :: IO ()
demoMapReduce = do
  let docs =
        [ "haskell is great haskell is pure"
        , "pure functional programming is great"
        , "haskell has great type system"
        ]
  let counts = wordCount docs
  putStrLn "Word counts (MapReduce):"
  mapM_ (\(w, c) -> putStrLn $ "  " ++ w ++ ": " ++ show c)
    $ Map.toAscList counts

demoMapReduce

Line 6: Eta reduce
Found:
mapStep f = concatMap f
Why not:
mapStep = concatMapLine 12: Eta reduce
Found:
reduceStep f = Map.mapWithKey f
Why not:
reduceStep = Map.mapWithKeyLine 15: Eta reduce
Found:
mapReduce mapper reducer input
  = reduceStep reducer . shuffleStep . mapStep mapper $ input
Why not:
mapReduce mapper reducer
  = reduceStep reducer . shuffleStep . mapStep mapperLine 21: Avoid lambda
Found:
\ doc -> map (\ w -> (w, 1)) (words doc)
Why not:
map (\ w -> (w, 1)) . wordsLine 21: Use tuple-section
Found:
\ w -> (w, 1)
Why not:
(, 1)

Word counts (MapReduce):
  functional: 1
  great: 3
  has: 1
  haskell: 3
  is: 3
  programming: 1
  pure: 2
  system: 1
  type: 1

## 🔵Категорный взгляд: единая картина

![Kategornyj vzglyad](../diagrams/haskell/dist_table.svg)

**Ключевые идеи:**
- CRDT = join-полурешётка: схождение гарантировано алгебраически
- Process монада = kleisli-стрелка: `a -> Process b`
- MapReduce = функтор + катаморфизм
- TChan/TQueue = канал в категории Proc

![Ландшафт распределённых вычислений](../diagrams/haskell/dist_landscape.svg)

---

**← Предыдущий:** [Конкурентность](Concurrency.ipynb)  |  **Следующий →** [GPU / Accelerate](GPUHaskell.ipynb)
